In [44]:
!pip install pandas numpy
!pip install scikit-learn
!pip install streamlit




  Using cached streamlit-1.58.0-py3-none-any.whl.metadata (9.6 kB)
  Using cached altair-6.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.1.4-py3-none-any.whl.metadata (5.5 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using cached pydeck-0.9.2-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached starlette-1.2.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached uvicorn-0.48.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached httptools-0.8.0-cp313-cp313-win_amd64.whl.metadata (3.7 kB)
  Using cached websockets-16.0-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl

In [45]:
import pandas as pd
import numpy as np

movies = pd.read_csv('C:\\Users\\user\\OneDrive\\Desktop\\movie-recommendation-system\\dataset\\tmdb_5000_movies.csv')
credits = pd.read_csv('C:\\Users\\user\\OneDrive\\Desktop\\movie-recommendation-system\\dataset\\tmdb_5000_credits.csv')


In [46]:
movies.head()
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [47]:
movies = movies.merge(credits, on='title')

In [48]:
movies = movies[[
    'movie_id',
    'title',
    'overview',
    'genres',
    'keywords',
    'cast',
    'crew'
]]

movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [49]:
movies.dropna(inplace=True)

In [50]:
# Convert json-like text
import ast

def convert(text):
    L = []

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [51]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [52]:
def convert_cast(text):
    L = []

    counter = 0

    for i in ast.literal_eval(text):

        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [53]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [54]:
#process director
def fetch_director(text):
    L = []

    for i in ast.literal_eval(text):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [55]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [56]:
#convert overview to list 
movies['overview'] = movies['overview'].apply(lambda x: x.split())


In [57]:
#create tag column 
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['crew']
)


In [58]:
#create final dataset
new_df = movies[['movie_id', 'title', 'tags']]

In [59]:
#convert list to string 
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [60]:
#text vectorization
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [61]:
#cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
#This is the main AI recommendation logic.

In [62]:
#create recommendation function
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [63]:
recommend('Avatar')

Aliens
Moonraker
Alien
Alien³
Silent Running


In [64]:
import streamlit as st

st.title("Movie Recommendation System")

movie_name = st.text_input("Enter Movie Name")

if st.button("Recommend"):

    recommend(movie_name)

2026-06-02 10:31:45.317 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 10:31:46.135 
  command:

    streamlit run c:\Users\user\OneDrive\Desktop\movie-recommendation-system\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-02 10:31:46.138 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 10:31:46.140 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 10:31:46.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 10:31:46.145 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 10:31:46.147 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06